
# Agentic Multimodal Demo (Notebook)
Lightweight, agentic pipeline you can run step-by-step:
- **SeriesBuilder**: ordered people/things with dates (kings, CEOs, etc.) → SDXL portraits → poster
- **MapBuilder**: regions/points (Europe, US states, Canadian provinces, South America) → flags/icons → map

Repo layout assumption:
```
./agentic-multimodal.ipynb      # this notebook (root)
./assets/agentic_multimodal/src # your supporting modules (you can create them later)
```


In [1]:

# --- Optional installs (uncomment as needed) ---
# %pip install langgraph langchain pydantic diffusers transformers accelerate
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install pillow geopandas shapely requests matplotlib bs4 pyproj
# %pip install ipywidgets
# Note: On Macs, torch install line will differ (MPS). See PyTorch docs.


In [2]:

# --- bootstrap: mount src folder and sanity-check ---
from pathlib import Path
import sys

SRC = Path("./assets/agentic_multimodal/src").resolve()
SRC.mkdir(parents=True, exist_ok=True)  # ensure directory exists
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("SRC mounted at:", SRC)
from adapters import wikidata_series as wds

SRC mounted at: /Users/douglasdaly/Documents/GitHub/Generative-AI/assets/agentic_multimodal/src


## Render your code tree (nice in Markdown)

In [3]:

from IPython.display import Markdown, display
from io import StringIO

def tree_markdown(root, max_depth=6, ignore=("__pycache__", ".DS_Store", ".git")):
    root = Path(root)
    lines = [f"```{root.name}/"]
    def walk(p, prefix, depth):
        if depth > max_depth:
            return
        children = sorted([c for c in p.iterdir() if c.name not in ignore], key=lambda x: (x.is_file(), x.name.lower()))
        for i, c in enumerate(children):
            is_last = i == len(children)-1
            branch = "└── " if is_last else "├── "
            lines.append(prefix + branch + c.name)
            if c.is_dir():
                extension = "    " if is_last else "│   "
                walk(c, prefix + extension, depth + 1)
    walk(root, "", 1)
    lines.append("```")
    return "\n".join(lines)

md = tree_markdown("./assets/agentic_multimodal/src")
display(Markdown(md))


```src/
├── adapters
│   ├── __init__.py
│   ├── natural_earth.py
│   ├── wikidata_geo.py
│   └── wikidata_series.py
├── agents
│   ├── __init__.py
│   ├── compositor.py
│   ├── image_gen.py
│   └── research.py
├── tasks
│   └── __init__.py
├── tools
│   ├── __init__.py
│   └── web.py
├── __init__.py
├── graph.py
└── schemas.py
```

## SeriesBuilder: Wikidata adapters (leaders, CEOs, etc.)

## Image generation (SDXL via Diffusers)

In [4]:
import requests
import time
import random
import os, re
from PIL import Image, ImageDraw, ImageFont
import math, torch

# Helpers to collect data

YEAR_RE = re.compile(r"(\d{4})")

def year_only(s):
    if not s:
        return ""
    m = YEAR_RE.search(s)
    return m.group(1) if m else s

UA = {"User-Agent":"agentic-multimodal/0.1 (+your_email@example.com)"}
YEAR = re.compile(r"(\d{3,4})")

def fetch_html(url, tries=5, base_sleep=0.6, timeout=45):
    for i in range(tries):
        r = None
        try:
            r = requests.get(url, headers=UA, timeout=timeout)
            r.raise_for_status()
            return r.text
        except requests.HTTPError:
            if r is not None and r.status_code in (403,429,500,502,503,504) and i < tries-1:
                time.sleep(min(base_sleep*(2**i)*(1+0.25*random.random()), 8.0))
                continue
            raise



In [5]:
# Set up diffusion model for image generation

import os, torch
from diffusers import StableDiffusionXLPipeline

def get_sdxl(device=None, model_id="stabilityai/stable-diffusion-xl-base-1.0"):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    dtype = torch.float16 if device=="cuda" else torch.float32
    pipe = StableDiffusionXLPipeline.from_pretrained(model_id, torch_dtype=dtype)
    pipe.to(device)
    pipe.enable_attention_slicing()
    return pipe, device




## Poster compositor (grid + captions)

In [6]:
# Generate collection of portraits
import os
from PIL import ImageFont

def resolve_font(preferred: str | None = None):
    """
    Return a path or font name that ImageFont.truetype can open.
    Tries: user-supplied path → common system fonts → Pillow's bundled DejaVu.
    """
    # 0) user-supplied path, if valid
    if preferred and os.path.isfile(preferred):
        return preferred

    # 1) common system fonts (macOS / Linux / Windows)
    candidates = [
        # macOS
        "/Library/Fonts/Arial.ttf",
        "/Library/Fonts/HelveticaNeue.ttf",
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Times New Roman.ttf",
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c

    # 2) Pillow ships DejaVuSans; most installs can resolve it by name
    try:
        # This works if PIL packaged fonts are on the font path
        ImageFont.truetype("DejaVuSans.ttf", size=10)
        return "DejaVuSans.ttf"
    except Exception:
        pass

    # 3) Absolute worst-case: raise a helpful error
    raise FileNotFoundError(
        "No usable TTF font found. Put a .ttf in assets/fonts/ and set base_font_path to it."
    )


def _safe_filename(s: str) -> str:
    s = re.sub(r"[^\w\-. ]+", "_", s)
    return s.strip("_")

def batch_generate_portraits(
    series_payload: dict,
    outdir: str = "out/series",
    steps: int = 25,
    scale: float = 6.5,
    size: tuple[int,int] = (512, 768),
    seed: int | None = 1337,     # set to None for non-deterministic
    skip_existing: bool = True,
):
    os.makedirs(outdir, exist_ok=True)
    pipe, device = get_sdxl()

    generator = None
    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    paths = []
    W, H = size  # (width, height)

    for i, item in enumerate(series_payload.get("items", []), 1):
        name  = item.get("name", f"item_{i}")
        prompt = item.get("prompt")
        neg_prompt = item.get("neg_prompt")

        fname = f"{i:03d}_{_safe_filename(name)}.png"
        fpath = os.path.join(outdir, fname)
        paths.append(fpath)

        if skip_existing and os.path.exists(fpath):
            continue

        img = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt,
            num_inference_steps=steps,
            guidance_scale=scale,
            width=W,
            height=H,
            generator=generator
        ).images[0]
        img.save(fpath)

    return paths



from PIL import Image, ImageDraw, ImageFont

def draw_centered_multiline(
    canvas: Image.Image,
    text_lines: list[str],
    center_xy: tuple[int,int],
    base_font_path: str | None = None,  # can be None now
    base_font_px: int = 28,
    fill=(0,0,0),
    stroke_fill=(255,255,255),
    stroke_width_px: int = 2,
    line_gap_px: int = 6,
    scale: float = 1.0,
):
    draw = ImageDraw.Draw(canvas)
    fsize = max(1, int(base_font_px * scale))
    gap   = max(1, int(line_gap_px * scale))
    sw    = max(1, int(stroke_width_px * scale))

    font_path = resolve_font(base_font_path) 
    font = ImageFont.truetype(font_path, fsize)
    # measure block
    widths, heights = [], []
    for line in text_lines:
        bbox = draw.textbbox((0,0), line, font=font, stroke_width=sw)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        widths.append(w); heights.append(h)
    block_w = max(widths) if widths else 0
    block_h = sum(heights) + gap * (len(text_lines) - 1 if text_lines else 0)

    cx, cy = center_xy
    x0 = int(cx - block_w/2)
    y0 = int(cy - block_h/2)

    y = y0
    for line, h in zip(text_lines, heights):
        draw.text((x0, y), line, font=font, fill=fill,
                  stroke_width=sw, stroke_fill=stroke_fill, align="center")
        y += h + gap


# --- Centered poster compositor with year-only dates ---

from math import ceil
from PIL import Image, ImageDraw, ImageFont

def compose_poster(
    series,
    image_paths,
    cols=6,
    size=(512, 768),              # (img_w, img_h) per tile
    margin=40,                    # outer margin
    gutter=24,                    # space between tiles
    base_font_path="assets/fonts/Inter-SemiBold.ttf",
    base_font_px=26,
    caption_scale=2.0,            # 2.0 = double-sized text
    line_gap_px=6,
    stroke_width_px=3,
    outpath=None,
):
    img_w, img_h = size
    n = len(image_paths)
    rows = ceil(n / cols)

    # Reserve vertical space for two caption lines below each image
    # (rough estimate; fine-tune if you want exact text bbox measurement)
    line_h = int(base_font_px * caption_scale)
    caption_h = line_h * 2 + int(line_gap_px * caption_scale)

    canvas_w = margin*2 + cols*img_w + (cols-1)*gutter
    canvas_h = margin*2 + rows*(img_h + caption_h) + (rows-1)*gutter
    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")

    def years_line(item):
        s = str(item.get("start") or "").strip()
        e = str(item.get("end") or "").strip()
        if not s and not e: return ""
        return f"{s} / {e or 'Present'}"

    k = 0
    for r in range(rows):
        for c in range(cols):
            if k >= n: break

            # --- This is where tile_x / tile_y come from ---
            tile_x = margin + c * (img_w + gutter)
            tile_y = margin + r * (img_h + caption_h + gutter)

            # paste image
            im = Image.open(image_paths[k]).convert("RGB").resize((img_w, img_h))
            canvas.paste(im, (tile_x, tile_y))

            # caption (centered under the image)
            item = series["items"][k]
            name = item.get("name", "")
            yrs  = years_line(item)

            caption_center = (tile_x + img_w // 2, tile_y + img_h + caption_h // 2)

            draw_centered_multiline(
                canvas,
                [name, yrs],
                center_xy=caption_center,
                base_font_path=base_font_path,
                base_font_px=base_font_px,
                stroke_width_px=stroke_width_px,
                line_gap_px=line_gap_px,
                scale=caption_scale,   # doubles text size & spacing
            )

            k += 1

    if outpath:
        canvas.save(outpath)
    return canvas




## MapBuilder: regions + capitals + flags

In [8]:

from assets.agentic_multimodal.src.adapters.wikidata_geo import get_points_for_region, render_region_map

#base, pts, label = get_points_for_region("Europe")
#out = render_region_map(base, pts, outpath="out/europe_flags.png", title=f"{label}: Capitals & Flags")


### Demo: Series poster — POTUS, British monarchs, Nobel Prize winners
#### Collect data

In [9]:
# POTUS
from assets.agentic_multimodal.src.adapters.wikidata_series import (
    series_monarchs_eng_gb_uk, series_nobel, series_potus
)
series = series_potus()


In [10]:
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/potus", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=8,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/potus_768_1024.png",
)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [11]:

# British monarchs (full stretch)
series = series_monarchs_eng_gb_uk()
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/monarchs", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=8,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/monarchs_england_great_britain_768_1024.png",
)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [12]:
# Nobel prize winners in Physics

series = series_nobel('Q38104', 'Physics')
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/physics_nobel", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=12,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/physics_nobel_768_1024.png",
)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:

if 0:
    # 2) Or, Nobel laureates in Physics
    #    (Q38104 = Nobel Prize in Physics)
    physics = series_nobel("Q38104", "Physics")

    # Render either with your existing pipeline:
    paths = batch_generate_portraits(series, outdir="results/monarchs", steps=20, scale=6.0)
    poster = compose_poster(series, paths, cols=12, size=(768,1024),
                            base_font_path=None, base_font_px=26, caption_scale=2.0,
                            outpath="results/monarchs_poster.png")


### Demo: Series poster — CEOs of General Electric

In [ ]:

# Resolve "General Electric" QID
wd_search_label("General Electric")[:3]


In [ ]:

GE_QID = "Q7729"  # verify with search above
series_ge = series_ceos_of_company(GE_QID, title="CEOs of GE (alpha template)")
len(series_ge["items"]), series_ge["items"][:3]


In [ ]:

paths_ge = batch_generate_portraits(series_ge, outdir="out/ge_ceos", steps=18, scale=6.0)
poster_ge = compose_poster(series_ge, paths_ge, cols=5, outpath="out/ge_ceos_poster.png")
poster_ge


### Demo: Map — Europe with capitals and flags

In [ ]:

world = load_admin0()
europe = filter_continent(world, "Europe")
df_caps = wikidata_capitals_for_continent("Q46")  # Europe
out_map = render_region_map(europe, df_caps, outpath="out/europe_map.png", epsg=3857)
out_map


### Demo: Map — South America

In [ ]:

south_america = filter_continent(world, "South America")
df_caps_sa = wikidata_capitals_for_continent("Q18")  # South America
out_map_sa = render_region_map(south_america, df_caps_sa, outpath="out/south_america_map.png", epsg=3857)
out_map_sa


### Demo: Map — US states (admin-1 centroids)

In [ ]:

admin1 = load_admin1()
us_states = filter_admin1_by_country(admin1, "United States")
# Compute approximate state centroids and label them; subnational flags need a different source (P41), so skip flags here.
centroids = us_states.to_crs(4326).copy()
centroids["lon"] = centroids.geometry.centroid.x
centroids["lat"] = centroids.geometry.centroid.y
pts = centroids[["name","lat","lon"]].rename(columns={"name":"country"})
pts["capital"] = ""  # placeholder
pts["iso2"] = ""     # no flags for subnationals via FlagCDN
out_us = render_region_map(us_states, pts, outpath="out/us_states_map.png", epsg=5070)  # Albers USA
out_us


## Re-render your `src` tree after you add files

In [ ]:

md = tree_markdown("./assets/agentic-multimodal/src")
display(Markdown(md))


# North Star

## Goal (1 sentence)
<What this system must do. No ANDs.>

## Non-Goals
- <Thing we are NOT solving>
- <Another non-goal>

## Primary Users & Usage
- **Who:** <persona/role>
- **How used:** <CLI/API/UI, frequency>
- **Success signal:** <what “done right” looks like>

## Public API Surface (contracts only)
- **Module/Service A**
  - `fn(sig) -> type` — <brief contract/invariants>
- **Module/Service B**
  - `endpoint METHOD /path` — <request/response shape>

## Invariants (must always hold)
- <e.g., “clockwise is UR→DR→DL→UL”>
- <e.g., “idempotent writes,” “no network in hot path,” “inputs validated at edge”>

## Performance/Scale Targets
- P50/P95 latency: <X ms>
- Memory cap: <Y MB>
- Throughput: <Z ops/sec>
- Max data size: <N>

## Extensibility Points
- <Where new features plug in—hooks, interfaces, events>

## Test Strategy
- Acceptance tests: <goldens you’ll keep stable>
- Contract tests: <per public interface>
- Property tests (if any): <invariants>

## Observability
- Log schema: `{event, component, id, elapsed_ms, ...}`
- Trace/Correlation: <how you propagate IDs>
- Metrics: <what you emit>

## Risks & Open Questions
- <Biggest unknowns to resolve>


# Decision: <short>
## Context
- <why change>
## Options
- A: <one line>
- B: <one line>
## Decision
- <chosen option>
## Consequences
- + <pros>
- − <cons>
## Migration/Notes
- <follow-ups, flags, rollbacks>


In [21]:
name = "this and that"
name = name.split(" and ", 1)[0].strip()
print(name)

this
